# Solving three Poisson problems in 1D/2D/3D

In [ ]:
#    APM41012EP course notebook - Chapter 5 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Iterative methods for solving linear systems
#    Poisson 1D, 2D, 3D
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
import time
import numpy as np

from scipy.sparse import diags
from scipy.sparse.linalg import spsolve, cg, norm

import matplotlib.pyplot as plt 
import plotly.graph_objects as go

## Poisson equation

We want to solve the elliptic problem given by the Poisson equation subject to Dirichlet boundary conditions:

$$
\left\{
\begin{aligned}
-\Delta u & =  f & \text{in} \; \Omega  \\
        u & =  g & \text{on}  \;  \partial \Omega
\end{aligned}
\right.
$$

## Structure of the 1d, 2d and 3d matrices for a fixed matrix size

In [ ]:
nx = 125
dx = 1/(nx+1)
diag = np.repeat(2/dx**2, nx)
diag_x = np.repeat(-1/dx**2, nx-1)
a_1d  = diags([diag, diag_x, diag_x], [0, -1, 1])

nx = 11
ny = 11
dx = 1/(nx+1)
dy = 1/(ny+1)
diag = np.repeat(2/dx**2 + 2/dy**2, nx*ny)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny)
diag_y = np.repeat(-1/dy**2, nx*(ny-1))
a_2d = diags([diag, diag_x, diag_x, diag_y, diag_y], [0, -1, 1, -nx, nx])

nx = 5
ny = 5
nz = 5
dx = 1/(nx+1)
dy = 1/(ny+1)
dz = 1/(nz+1)
diag = np.repeat(2/dx**2 + 2/dx**2 + 2/dz**2, nx*ny*nz)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny*nz)
diag_y = np.tile(np.repeat([-1/dy**2, 0.], (nx*(ny-1), nz)), nz)
diag_z = np.repeat(-1/dz**2, nx*ny*(nz-1))
a_3d = diags([diag, diag_x, diag_x, diag_y, diag_y, diag_z, diag_z], [0, -1, 1, -nx, nx, -nx*ny, nx*ny])

plt.figure(figsize=[18,6])
ax1 = plt.subplot(1, 3, 1)
ax1.set_title('1d case')
plt.spy(a_1d, markersize=4)

ax2 = plt.subplot(1, 3, 2)
ax2.set_title('2d case')
plt.spy(a_2d, markersize=4) 

ax3 = plt.subplot(1, 3, 3)
ax3.set_title('3d case')
plt.spy(a_3d, markersize=4) 

plt.show()

## Eigenvalues of the Laplacian

In [ ]:
def eig_val_1d(nx, dx, i):
    a = (4/(dx**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2

def eig_val_2d(nx, dx, ny, dy, i, j):
    a = (4/(dx**2))
    b = (4/(dy**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2 + b*np.sin((np.pi*j)/(2*(ny+1)))**2

def eig_val_3d(nx, dx, ny, dy, nz, dz, i, j, k):
    a = (4/(dx**2))
    b = (4/(dy**2))
    c = (4/(dz**2))
    return a*np.sin((np.pi*i)/(2*(nx+1)))**2 + b*np.sin((np.pi*j)/(2*(ny+1)))**2 + c*np.sin((np.pi*k)/(2*(nz+1)))**2

In [ ]:
def conjugate_gradient(a, b, tol=1e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        hist_norm_rk.append(norm_rk/norm_b)
        if norm_rk/norm_b < tol: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    #print(f"  Number of iterations = {k+1}")
    #print(f"  ||A.xk - b|| / ||b|| = {norm_rk/norm_b}")

    return xk, hist_norm_rk

def conjugate_gradient_bis(a, b, xexa, tol=1e-6):
    xk = np.zeros(b.size)
    norm_b = np.linalg.norm(b)

    rk = b - a.dot(xk)
    pk = rk
    rkm1 = rk
    
    hist_norm_rk = []
    hist_norm_rk.append(1.0)
    hist_norm_err = []
    hist_norm_err.append(1.0)
    err0 = np.dot(xexa, a.dot(xexa))

    for k in range(b.size):
        apk = a.dot(pk)
        alpha = np.dot(rk,rk) / np.dot(pk, apk)
        xk = xk + alpha*pk
        rk = rk - alpha*apk
        norm_rk = np.linalg.norm(rk)
        hist_norm_rk.append(norm_rk/norm_b)
        err = xexa - xk
        hist_norm_err.append(np.sqrt(np.dot(err, a.dot(err))/err0))
        if norm_rk/norm_b < tol: break
        beta = np.dot(rk,rk) / np.dot(rkm1, rkm1)
        pk = rk + beta*pk
        rkm1 = rk

    print(f"  Number of iterations = {k+1}")
    print(f"  ||rk|| / ||b||                      = {norm_rk/norm_b}")

    return xk, hist_norm_rk, hist_norm_err

## 1d case

$$
\left\{
\begin{aligned}
- & u''(x)  =  f(x) \quad \text{in} \; \Omega = [0,1] \quad \text{with} \; f(x)=1\\
  & u(0)  =  0 \; \text{and} \; u(1) = 0
\end{aligned}
\right.
$$

In [ ]:
nx = 122500
dx = 1/(nx+1)

# building the sparse matrix
diag = np.repeat(2/dx**2, nx)
diag_x = np.repeat(-1/dx**2, nx-1)
a  = diags([diag, diag_x, diag_x], [0, -1, 1])
norm_a = norm(a)

# right-hand side
b = np.ones(nx)

print(f"1d case: nx = {nx}")
print(f"Matrix size: ({nx} x {nx})")
print(f"Bandwidth: {3}")
print(f"Entries of the factorised matrix: {nx*3:,}".replace(',',' '))
print(f"Condition number: {eig_val_1d(nx, dx, nx)/eig_val_1d(nx, dx, 1)}")

t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time()
print("\nSolution using a direct method")
res = np.linalg.norm(b - a.dot(ulu))
print(f"  ||A.xk - b|| / ||b||                = {res/np.linalg.norm(b)}")
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")
print(f"Execution time (s): {t2-t1}")

print("\nSolution using the conjugate gradient method")
ucg, hist_cg_1d  = conjugate_gradient(a, b)
t1 = time.time()   
ucg, hist_cg_1d, hist_err_1d  = conjugate_gradient_bis(a, b, ucg)
t2 = time.time()
nit_cg_1d = len(hist_cg_1d)
res = np.linalg.norm(b - a.dot(ucg))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ucg) + np.linalg.norm(b))}")
print(f"Execution time (s): {t2-t1}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nit_cg_1d), y=hist_err_1d, name="Error in A-norm"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_1d), y=hist_cg_1d, name="||rk|| / ||b||"))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e')
fig.update_layout(title="Convergence history", legend = dict(orientation="h", y=1.1))
fig.show()

## 2d case

$$
\left\{
\begin{aligned}
-\Delta u(x,y) & =  f(x,y) \quad \text{ in } \; \Omega = [0,1] \times [0,1]  \quad \text{with} \; f(x)=1 \\
        u(x,y) & =  0 \quad \text{ on }  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
nx = 350
ny = 350
dx = 1/(nx+1)
dy = 1/(ny+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dy**2, nx*ny)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny)
diag_y = np.repeat(-1/dy**2, nx*(ny-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y], [0, -1, 1, -nx, nx])
norm_a = norm(a)

# right-hand side
b = np.ones(nx*ny)

print(f"2d case: nx = {nx} and ny = {ny} => nx . ny = {nx*ny}")
print(f"Matrix size: ({nx*ny} x {nx*ny})")
print(f"Bandwidth: {2*nx}")
print(f"Entries of the factorised matrix: {nx*ny*(2*nx):,}".replace(',',' '))
print(f"Condition number: {eig_val_2d(nx, dx, ny, dy, nx, ny)/eig_val_2d(nx, dx, ny, dy, 1, 1)}")

t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time()
print("\nSolution using a direct method")
res = np.linalg.norm(b - a.dot(ulu))
print(f"  ||A.xk - b|| / ||b||                = {res/np.linalg.norm(b)}")
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")
print(f"Execution time: {t2-t1}")

print("\nSolution using the conjugate gradient method")
ucg, hist_cg_2d  = conjugate_gradient(a, b, tol=1e-10)
t1 = time.time() 
ucg, hist_cg_2d, hist_err_2d  = conjugate_gradient_bis(a, b, ucg)
t2 = time.time()
res = np.linalg.norm(b - a.dot(ucg))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ucg) + np.linalg.norm(b))}")
print(f"Execution time: {t2-t1}")

nit_cg_2d = len(hist_cg_2d)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nit_cg_2d), y=hist_err_1d[:nit_cg_2d], name="Error in A-norm 1d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_2d), y=hist_cg_1d[:nit_cg_2d], name="||rk|| / ||b|| 1d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_2d), y=hist_err_2d, name="Error in A-norm 2d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_2d), y=hist_cg_2d, name="||rk|| / ||b|| 2d"))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e')
fig.update_layout(title="Convergence history")
fig.show()

## 3d case

$$
\left\{
\begin{aligned}
-\Delta u(x,y,z) & =  f(x,y,z) \quad \text{in} \; \Omega = [0,1] \times [0,1] \times [0,1] \quad \text{with} \; f(x)=1\\
        u(x,y,z) & =  0 \quad \text{on}  \;  \partial \Omega
\end{aligned}
\right.
$$

In [ ]:
nx = 50
ny = 50
nz = 49
dx = 1/(nx+1)
dy = 1/(ny+1)
dz = 1/(nz+1)

# building the sparse matrix
diag = np.repeat(2/dx**2 + 2/dx**2 + 2/dz**2, nx*ny*nz)
diag_x = np.tile(np.repeat([-1/dx**2, 0.], (nx-1, 1)), ny*nz)
diag_y = np.tile(np.repeat([-1/dy**2, 0.], (nx*(ny-1), nz)), nz)
diag_z = np.repeat(-1/dz**2, nx*ny*(nz-1))
a = diags([diag, diag_x, diag_x, diag_y, diag_y, diag_z, diag_z], [0, -1, 1, -nx, nx, -nx*ny, nx*ny])
norm_a = norm(a)

# right-hand side
b = np.ones(nx*ny*nz)

print(f"3d case: nx = {nx}, ny = {ny} and nz = {nz} => nx . ny . nz = {nx*ny*nz}")
print(f"Matrix size: ({nx*ny*nz} x {nx*ny*nz})")
print(f"Bandwidth: {2*nx*ny}")
print(f"Entries of the factorised matrix: {nx*ny*nz*(2*nx*ny):,}".replace(',',' '))
print(f"Condition number: {eig_val_3d(nx, dx, ny, dy, nz, dz, nx, ny, nz)/eig_val_3d(nx, dx, ny, dy, nz, dz, 1, 1, 1)}")

print("\nSolution using a direct method")
t1 = time.time()    
ulu = spsolve(a.tocsr(), b)
t2 = time.time() 
res = np.linalg.norm(b - a.dot(ulu))
print(f"  ||A.xk - b|| / ||b||                = {res/np.linalg.norm(b)}")
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ulu) + np.linalg.norm(b))}")
print(f"Execution time: {t2-t1}")

print("\nSolution using the conjugate gradient method")
ucg, hist_cg_3d  = conjugate_gradient(a, b, tol=1e-10)
t1 = time.time()
ucg, hist_cg_3d, hist_err_3d  = conjugate_gradient_bis(a, b, ucg)
t2 = time.time()
res = np.linalg.norm(b - a.dot(ucg))
print(f"  ||A.xk - b|| / ||A|| ||xk|| + ||b|| = {res / (norm_a * np.linalg.norm(ucg) + np.linalg.norm(b))}")
print(f"Execution time: {t2-t1}")

nit_cg_3d = len(hist_cg_3d)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_err_1d[:nit_cg_3d], name="Error in A-norm 1d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_cg_1d[:nit_cg_3d], name="||rk|| / ||b|| 1d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_err_2d[:nit_cg_3d], name="Error in A-norm 2d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_cg_2d[:nit_cg_3d], name="||rk|| / ||b|| 2d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_err_3d[:nit_cg_3d], name="Error in A-norm 3d"))
fig.add_trace(go.Scatter(x=np.arange(nit_cg_3d), y=hist_cg_3d[:nit_cg_3d], name="||rk|| / ||b|| 3d"))
fig.update_xaxes(title="number of iterations")
fig.update_yaxes(type="log", exponentformat = 'e')
fig.update_layout(title="Convergence history")
fig.show()